# Problem 2 - Transformer from scratch

Complete implementation of restricted attention and a causal character Transformer.

In [ ]:
from pathlib import Path
import random,time,urllib.request,math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
ROOT=Path.cwd().parent if Path.cwd().name=='problem2' else Path.cwd(); FIGURES=ROOT/'problem2'/'figures'; FIGURES.mkdir(parents=True,exist_ok=True); DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(DEVICE)

In [ ]:
def scaled_dot_product_attention(q,k,v,mask=None):
    scores=(q@k.transpose(-2,-1))/math.sqrt(q.size(-1))
    if mask is not None: scores=scores.masked_fill(~mask,-torch.finfo(scores.dtype).max) if mask.dtype==torch.bool else scores+mask
    weights=F.softmax(scores,dim=-1); return weights@v,weights
class MyMultiHeadAttention(nn.Module):
    def __init__(self,d_model,heads):
        super().__init__(); assert d_model%heads==0,'d_model must be divisible by heads'; self.d_model=d_model; self.heads=heads; self.head_dim=d_model//heads
        self.q_proj=nn.Linear(d_model,d_model); self.k_proj=nn.Linear(d_model,d_model); self.v_proj=nn.Linear(d_model,d_model); self.o_proj=nn.Linear(d_model,d_model)
    def forward(self,x,mask=None):
        b,t,_=x.shape; split=lambda z:z.view(b,t,self.heads,self.head_dim).transpose(1,2); y,w=scaled_dot_product_attention(split(self.q_proj(x)),split(self.k_proj(x)),split(self.v_proj(x)),mask); return self.o_proj(y.transpose(1,2).contiguous().view(b,t,self.d_model)),w
class EncoderBlock(nn.Module):
    def __init__(self,d_model,heads,dropout=0.1):
        super().__init__(); self.norm1=nn.LayerNorm(d_model); self.attn=MyMultiHeadAttention(d_model,heads); self.norm2=nn.LayerNorm(d_model); self.ff=nn.Sequential(nn.Linear(d_model,4*d_model),nn.GELU(),nn.Linear(4*d_model,d_model)); self.dropout=nn.Dropout(dropout)
    def forward(self,x,mask=None):
        y,w=self.attn(self.norm1(x),mask); x=x+self.dropout(y); return x+self.dropout(self.ff(self.norm2(x))),w
q,k,v=torch.randn(2,4,8,16),torch.randn(2,4,8,16),torch.randn(2,4,8,16); _,w=scaled_dot_product_attention(q,k,v); assert torch.allclose(w.sum(-1),torch.ones_like(w.sum(-1)),atol=1e-5); print('attention checks passed; weight-only block count:',4*128**2+2*128*512)

In [ ]:
SHAKESPEARE_URL='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
def load_text(cache_path=ROOT/'problem2'/'tinyshakespeare.txt',timeout=30):
    cache_path.parent.mkdir(parents=True,exist_ok=True)
    if cache_path.exists(): text=cache_path.read_text(encoding='utf-8')
    else:
        with urllib.request.urlopen(SHAKESPEARE_URL,timeout=timeout) as response: text=response.read().decode('utf-8'); cache_path.write_text(text,encoding='utf-8')
    if len(text)<=1: raise ValueError('Tiny Shakespeare data is empty')
    chars=sorted(set(text)); stoi={c:i for i,c in enumerate(chars)}; ids=np.array([stoi[c] for c in text],dtype=np.int64); cut=max(1,min(len(ids)-1,int(.9*len(ids)))); return text,chars,torch.tensor(ids[:cut]),torch.tensor(ids[cut:])
class CharTransformer(nn.Module):
    def __init__(self,vocab_size,d_model=128,heads=4,layers=4,context=128,dropout=.1):
        super().__init__(); self.context=context; self.tok=nn.Embedding(vocab_size,d_model); self.pos=nn.Embedding(context,d_model); self.blocks=nn.ModuleList([EncoderBlock(d_model,heads,dropout) for _ in range(layers)]); self.norm=nn.LayerNorm(d_model); self.lm=nn.Linear(d_model,vocab_size,bias=False); self.lm.weight=self.tok.weight
    def forward(self,idx,targets=None,mask=None,return_weights=False):
        if idx.ndim!=2 or idx.dtype not in (torch.int64,torch.long): raise ValueError('idx must be an integer tensor of shape (batch,time)')
        b,t=idx.shape; assert t<=self.context and t>0; x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device)); mask=torch.tril(torch.ones(t,t,dtype=torch.bool,device=idx.device))[None,None] if mask is None else mask; ws=[]
        for block in self.blocks: x,w=block(x,mask); ws.append(w)
        logits=self.lm(self.norm(x)); loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),targets.reshape(-1)) if targets is not None else None; return (logits,loss,ws) if return_weights else (logits,loss)
def batches(ids,context,batch_size,device=DEVICE):
    if ids.ndim!=1 or context<1 or len(ids)<context+1 or batch_size<1: raise ValueError('ids, context, or batch_size is invalid')
    starts=torch.randint(0,len(ids)-context,(batch_size,)); x=torch.stack([ids[i:i+context] for i in starts]); y=torch.stack([ids[i+1:i+context+1] for i in starts]); return x.to(device),y.to(device)
def train_lm(model,train_ids,val_ids,steps=2000,batch_size=64,lr=3e-4,eval_every=100,patience=8):
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=.01); sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,steps); hist={'step':[],'train':[],'validation':[]}; best=float('inf'); best_state=None; stale=0; t0=time.perf_counter()
    for step in range(1,steps+1):
        model.train(); x,y=batches(train_ids,model.context,batch_size); _,loss=model(x,y); opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); sched.step()
        if step==1 or step%eval_every==0 or step==steps:
            model.eval(); x,y=batches(val_ids,model.context,batch_size);
            with torch.no_grad(): _,vl=model(x,y)
            value=vl.item(); hist['step'].append(step); hist['train'].append(loss.item()); hist['validation'].append(value)
            if value<best: best=value; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
            else: stale+=1
            if stale>=patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return hist,(time.perf_counter()-t0)/max(step,1)
def sample(model,seed_ids,chars,n=500,temperature=.8,top_k=20):
    if temperature<=0 or top_k<1: raise ValueError('temperature must be positive and top_k must be positive')
    model.eval(); ids=seed_ids.flatten().clone().to(DEVICE)
    with torch.no_grad():
        for _ in range(n):
            logits,_=model(ids[-model.context:].unsqueeze(0)); vals,inds=torch.topk(logits[0,-1]/temperature,min(top_k,logits.size(-1))); ids=torch.cat((ids,inds[torch.multinomial(F.softmax(vals,0),1)]))
    return ''.join(chars[i] for i in ids[-n:].cpu().tolist())
def plot_curves(hist,path):
    fig,ax=plt.subplots(); ax.plot(hist['step'],hist['train'],label='train'); ax.plot(hist['step'],hist['validation'],label='validation'); ax.legend(); fig.savefig(path,dpi=140); plt.close(fig)
def attention_heatmap(model,idx,path,layer=0,head=0):
    _,_,ws=model(idx[:,:model.context].to(DEVICE),return_weights=True); fig,ax=plt.subplots(); ax.imshow(ws[layer][0,head].detach().cpu(),aspect='auto'); fig.savefig(path,dpi=140); plt.close(fig)
def profile_contexts(model,ids,lengths=(32,64,128)):
    rows=[]
    for n in lengths:
        if n>model.context: continue
        x,y=batches(ids,n,1); t=time.perf_counter(); model(x,y); rows.append({'context':n,'seconds':time.perf_counter()-t,'parameters':sum(p.numel() for p in model.parameters())})
    return rows

## Full experiment (opt-in)
Set `RUN_EXPERIMENT=True` to download Tiny Shakespeare, train, generate 500-character samples at all required temperatures, save curves/heatmaps, and profile context lengths.

In [ ]:
def train_lm_accelerated(model,train_ids,val_ids,steps=10500,batch_size=128,lr=3e-4,eval_every=500,patience=8):
    use_amp=DEVICE.type=='cuda'; scaler=torch.cuda.amp.GradScaler(enabled=use_amp); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=.01); sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,steps); hist={'step':[],'train':[],'validation':[]}; best=float('inf'); best_state=None; stale=0; t0=time.perf_counter()
    for step in range(1,steps+1):
        model.train(); x,y=batches(train_ids,model.context,batch_size); opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,dtype=torch.float16 if use_amp else torch.float32,enabled=use_amp): _,loss=model(x,y)
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.); scaler.step(opt); scaler.update(); sched.step()
        if step==1 or step%eval_every==0 or step==steps:
            model.eval(); x,y=batches(val_ids,model.context,batch_size);
            with torch.no_grad(), torch.autocast(device_type=DEVICE.type,dtype=torch.float16 if use_amp else torch.float32,enabled=use_amp): _,vl=model(x,y)
            value=vl.item(); hist['step'].append(step); hist['train'].append(loss.item()); hist['validation'].append(value)
            if value<best: best=value; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
            else: stale+=1
            if stale>=patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return hist,(time.perf_counter()-t0)/max(step,1)

RUN_EXPERIMENT=False
if RUN_EXPERIMENT:
    seed_everything(0); torch.set_float32_matmul_precision('high'); text,chars,train_ids,val_ids=load_text(); model=CharTransformer(len(chars)).to(DEVICE)
    if DEVICE.type=='cuda':
        torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
        try: model=torch.compile(model,mode='reduce-overhead')
        except Exception as exc: print('torch.compile unavailable:',exc)
    hist,sec=train_lm_accelerated(model,train_ids,val_ids,steps=10500,batch_size=128,eval_every=500,patience=8); report_model=model._orig_mod if hasattr(model,'_orig_mod') else model; plot_curves(hist,FIGURES/'problem2_training.png')
    for temp in (.2,.8,1.5): print(f'\n--- temperature={temp} ---\n{sample(report_model,train_ids[:1],chars,500,temp,20)}')
    attention_heatmap(report_model,train_ids[:128].unsqueeze(0),FIGURES/'problem2_attention.png'); print(profile_contexts(report_model,val_ids)); print('parameters:',sum(p.numel() for p in report_model.parameters()))